# 05 ML GNN Embeddings — Group 1: Link-Prediction Embeddings (Log Target)

**Group 1.** Trains regressors on GNN embeddings produced by the link-prediction training objective — no reconstruction loss, no fine-tuning. Target is `log_systemic_risk_label`.

| Dataset | Model | Dim |
|---|---|---|
| `graphsage_v1_32_srisk_dataset.parquet` | GraphSAGE v1 | 32 |
| `graphsage_v1_64_srisk_dataset.parquet` | GraphSAGE v1 | 64 |
| `graphsage_v1_128_srisk_dataset.parquet` | GraphSAGE v1 | 128 |
| `node2vec_v1_32_srisk_dataset.parquet` | Node2Vec v1 | 32 |
| `node2vec_v1_64_srisk_dataset.parquet` | Node2Vec v1 | 64 |
| `node2vec_v1_128_srisk_dataset.parquet` | Node2Vec v1 | 128 |

> Run `03_g1_ref.ipynb` first to generate the parquet files.

In [1]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

sys.path.insert(0, os.path.abspath('../..'))
from src.models.ml_train_and_store import (
    ModelTrainer,
    load_gnn_dataset,
    make_pipeline,
)

pd.set_option("display.max_columns", 200)
PROJECT_ROOT = Path().resolve().parents[1]

In [2]:
print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


## Load Datasets

In [3]:
df_sage_32, feature_cols_sage_32 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="graphsage_v1_32_srisk_dataset.parquet")
print(f"GraphSAGE v1 32:  {df_sage_32.shape}  —  {len(feature_cols_sage_32)} embedding cols")

df_sage_64, feature_cols_sage_64 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="graphsage_v1_64_srisk_dataset.parquet")
print(f"GraphSAGE v1 64:  {df_sage_64.shape}  —  {len(feature_cols_sage_64)} embedding cols")

df_sage_128, feature_cols_sage_128 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="graphsage_v1_128_srisk_dataset.parquet")
print(f"GraphSAGE v1 128: {df_sage_128.shape}  —  {len(feature_cols_sage_128)} embedding cols")

df_n2v_32, feature_cols_n2v_32 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="node2vec_v1_32_srisk_dataset.parquet")
print(f"Node2Vec v1 32:   {df_n2v_32.shape}  —  {len(feature_cols_n2v_32)} embedding cols")

df_n2v_64, feature_cols_n2v_64 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="node2vec_v1_64_srisk_dataset.parquet")
print(f"Node2Vec v1 64:   {df_n2v_64.shape}  —  {len(feature_cols_n2v_64)} embedding cols")

df_n2v_128, feature_cols_n2v_128 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="node2vec_v1_128_srisk_dataset.parquet")
print(f"Node2Vec v1 128:  {df_n2v_128.shape}  —  {len(feature_cols_n2v_128)} embedding cols")

GraphSAGE v1 32:  (145536, 37)  —  32 embedding cols
GraphSAGE v1 64:  (145536, 69)  —  64 embedding cols
GraphSAGE v1 128: (145536, 133)  —  128 embedding cols
Node2Vec v1 32:   (145536, 37)  —  32 embedding cols
Node2Vec v1 64:   (145536, 69)  —  64 embedding cols
Node2Vec v1 128:  (145536, 133)  —  128 embedding cols


In [4]:
trainer_sage_32  = ModelTrainer(df=df_sage_32,  feature_cols=feature_cols_sage_32,  target_col="log_systemic_risk_label")
trainer_sage_64  = ModelTrainer(df=df_sage_64,  feature_cols=feature_cols_sage_64,  target_col="log_systemic_risk_label")
trainer_sage_128 = ModelTrainer(df=df_sage_128, feature_cols=feature_cols_sage_128, target_col="log_systemic_risk_label")
trainer_n2v_32   = ModelTrainer(df=df_n2v_32,   feature_cols=feature_cols_n2v_32,   target_col="log_systemic_risk_label")
trainer_n2v_64   = ModelTrainer(df=df_n2v_64,   feature_cols=feature_cols_n2v_64,   target_col="log_systemic_risk_label")
trainer_n2v_128  = ModelTrainer(df=df_n2v_128,  feature_cols=feature_cols_n2v_128,  target_col="log_systemic_risk_label")

print("GraphSAGE v1 32  —", trainer_sage_32.train_df.shape,  trainer_sage_32.val_df.shape)
print("GraphSAGE v1 64  —", trainer_sage_64.train_df.shape,  trainer_sage_64.val_df.shape)
print("GraphSAGE v1 128 —", trainer_sage_128.train_df.shape, trainer_sage_128.val_df.shape)
print("Node2Vec v1 32   —", trainer_n2v_32.train_df.shape,   trainer_n2v_32.val_df.shape)
print("Node2Vec v1 64   —", trainer_n2v_64.train_df.shape,   trainer_n2v_64.val_df.shape)
print("Node2Vec v1 128  —", trainer_n2v_128.train_df.shape,  trainer_n2v_128.val_df.shape)

GraphSAGE v1 32  — (109152, 37) (18192, 37)
GraphSAGE v1 64  — (109152, 69) (18192, 69)
GraphSAGE v1 128 — (109152, 133) (18192, 133)
Node2Vec v1 32   — (109152, 37) (18192, 37)
Node2Vec v1 64   — (109152, 69) (18192, 69)
Node2Vec v1 128  — (109152, 133) (18192, 133)


## Define Models

In [5]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]
TOP1_COLS    = ["model", "train_top1_mae", "validation_top1_mae", "train_top1_rmse", "validation_top1_rmse"]

candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}

list(candidate_models)

['Linear Regression',
 'Ridge',
 'MLP',
 'Random Forest',
 'Gradient Boosting',
 'XGBoost']

## Train — GraphSAGE v1 (32-dim)

In [6]:
trainer_sage_32.train_all(candidate_models)
display(trainer_sage_32.leaderboard()[DISPLAY_COLS])
trainer_sage_32.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,MLP,0.019889,0.027906,0.08647,0.127898
1,Random Forest,0.005616,0.024808,0.031387,0.138866
2,Gradient Boosting,0.01018,0.025387,0.055033,0.1561
3,XGBoost,0.009549,0.026002,0.050508,0.157843
4,Ridge,0.031269,0.039357,0.111204,0.170686
5,Linear Regression,0.031269,0.039358,0.111204,0.170688


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,MLP,0.484523,0.713926,0.591542,0.832469
1,Random Forest,0.167353,0.836358,0.205482,0.958688
2,Gradient Boosting,0.30016,0.968652,0.376001,1.099647
3,XGBoost,0.26941,0.986194,0.342126,1.113119
4,Ridge,0.626121,0.970383,0.780094,1.109962
5,Linear Regression,0.626107,0.970379,0.78008,1.109958


## Train — GraphSAGE v1 (64-dim)

In [7]:
trainer_sage_64.train_all(candidate_models)
display(trainer_sage_64.leaderboard()[DISPLAY_COLS])
trainer_sage_64.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Ridge,0.034163,0.037978,0.133128,0.160703
1,Linear Regression,0.034164,0.037982,0.133128,0.160712
2,Random Forest,0.006826,0.028328,0.038976,0.164166
3,Gradient Boosting,0.011686,0.029116,0.064773,0.188563
4,XGBoost,0.010257,0.029695,0.055726,0.194917
5,MLP,0.026224,0.03672,0.126762,0.207075


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Ridge,0.79614,0.849649,0.975195,0.988317
1,Linear Regression,0.796108,0.84966,0.975164,0.988335
2,Random Forest,0.195164,0.946436,0.246611,1.132305
3,Gradient Boosting,0.360066,1.147345,0.454093,1.340084
4,XGBoost,0.305483,1.155453,0.386165,1.384605
5,MLP,0.6436,1.135266,0.799591,1.431237


## Train — GraphSAGE v1 (128-dim)

In [8]:
trainer_sage_128.train_all(candidate_models)
display(trainer_sage_128.leaderboard()[DISPLAY_COLS])
trainer_sage_128.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.007059,0.029757,0.039749,0.177422
1,XGBoost,0.010159,0.030592,0.055731,0.194685
2,Gradient Boosting,0.009814,0.030842,0.056046,0.197051
3,Ridge,0.032702,0.040652,0.14169,0.200597
4,Linear Regression,0.032706,0.040661,0.14169,0.200627
5,MLP,0.039877,0.042195,0.845186,0.201919


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.200628,0.934088,0.249928,1.190614
1,XGBoost,0.30853,1.105169,0.391238,1.362728
2,Gradient Boosting,0.292059,1.124423,0.391781,1.38129
3,Ridge,0.87784,0.965246,1.081777,1.193157
4,Linear Regression,0.877798,0.965176,1.081735,1.193093
5,MLP,0.857749,1.019686,1.066389,1.247266


## Train — Node2Vec v1 (32-dim)

In [9]:
trainer_n2v_32.train_all(candidate_models)
display(trainer_n2v_32.leaderboard()[DISPLAY_COLS])
trainer_n2v_32.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.006602,0.029737,0.031777,0.128827
1,Gradient Boosting,0.012095,0.029378,0.057551,0.131989
2,XGBoost,0.013066,0.030195,0.061143,0.133008
3,MLP,0.026856,0.046704,0.083287,0.1448
4,Linear Regression,0.052624,0.077365,0.13278,0.191673
5,Ridge,0.052623,0.077363,0.13278,0.191674


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.179467,0.734742,0.220007,0.816724
1,Gradient Boosting,0.289365,0.781015,0.389576,0.850982
2,XGBoost,0.311242,0.791244,0.424293,0.855324
3,MLP,0.494593,0.827235,0.586607,0.899891
4,Linear Regression,0.858818,1.078168,0.984392,1.173783
5,Ridge,0.858836,1.078185,0.984408,1.173804


## Train — Node2Vec v1 (64-dim)

In [10]:
trainer_n2v_64.train_all(candidate_models)
display(trainer_n2v_64.leaderboard()[DISPLAY_COLS])
trainer_n2v_64.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.006424,0.027934,0.031188,0.127446
1,XGBoost,0.011768,0.028048,0.053679,0.130683
2,Gradient Boosting,0.01562,0.028518,0.071728,0.132775
3,MLP,0.025408,0.041764,0.069304,0.135194
4,Linear Regression,0.054328,0.075092,0.12928,0.201878
5,Ridge,0.054326,0.07509,0.12928,0.201878


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.17626,0.733163,0.215209,0.798935
1,XGBoost,0.263444,0.769678,0.35767,0.839067
2,Gradient Boosting,0.419656,0.809907,0.515262,0.873844
3,MLP,0.369121,0.753243,0.452768,0.840663
4,Linear Regression,0.824853,1.124582,0.938636,1.214111
5,Ridge,0.824863,1.124588,0.938645,1.214118


## Train — Node2Vec v1 (128-dim)

In [11]:
trainer_n2v_128.train_all(candidate_models)
display(trainer_n2v_128.leaderboard()[DISPLAY_COLS])
trainer_n2v_128.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.006331,0.027654,0.031798,0.121654
1,XGBoost,0.010873,0.029189,0.048699,0.130537
2,Gradient Boosting,0.011745,0.029,0.055493,0.130588
3,MLP,0.021324,0.043428,0.058999,0.14532
4,Ridge,0.053409,0.08326,0.128638,0.193767
5,Linear Regression,0.053411,0.083262,0.128638,0.193767


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.18056,0.637391,0.22062,0.713651
1,XGBoost,0.227172,0.688238,0.313251,0.765946
2,Gradient Boosting,0.283858,0.695074,0.369611,0.775445
3,MLP,0.286074,0.801616,0.379043,0.900057
4,Ridge,0.827239,1.076182,0.9589,1.197344
5,Linear Regression,0.827224,1.076172,0.958884,1.197329


## Top 1% Leaderboards

In [12]:
print("GraphSAGE v1 (32-dim)");  display(trainer_sage_32.leaderboard()[TOP1_COLS])
print("GraphSAGE v1 (64-dim)");  display(trainer_sage_64.leaderboard()[TOP1_COLS])
print("GraphSAGE v1 (128-dim)"); display(trainer_sage_128.leaderboard()[TOP1_COLS])
print("Node2Vec v1 (32-dim)");   display(trainer_n2v_32.leaderboard()[TOP1_COLS])
print("Node2Vec v1 (64-dim)");   display(trainer_n2v_64.leaderboard()[TOP1_COLS])
print("Node2Vec v1 (128-dim)");  display(trainer_n2v_128.leaderboard()[TOP1_COLS])

GraphSAGE v1 (32-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,MLP,0.484523,0.713926,0.591542,0.832469
1,Random Forest,0.167353,0.836358,0.205482,0.958688
2,Gradient Boosting,0.30016,0.968652,0.376001,1.099647
3,XGBoost,0.26941,0.986194,0.342126,1.113119
4,Ridge,0.626121,0.970383,0.780094,1.109962
5,Linear Regression,0.626107,0.970379,0.78008,1.109958


GraphSAGE v1 (64-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Ridge,0.79614,0.849649,0.975195,0.988317
1,Linear Regression,0.796108,0.84966,0.975164,0.988335
2,Random Forest,0.195164,0.946436,0.246611,1.132305
3,Gradient Boosting,0.360066,1.147345,0.454093,1.340084
4,XGBoost,0.305483,1.155453,0.386165,1.384605
5,MLP,0.6436,1.135266,0.799591,1.431237


GraphSAGE v1 (128-dim)


...

,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.179467,0.734742,0.220007,0.816724
1,Gradient Boosting,0.289365,0.781015,0.389576,0.850982
2,XGBoost,0.311242,0.791244,0.424293,0.855324
3,MLP,0.494593,0.827235,0.586607,0.899891
4,Linear Regression,0.858818,1.078168,0.984392,1.173783
5,Ridge,0.858836,1.078185,0.984408,1.173804


Node2Vec v1 (64-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.17626,0.733163,0.215209,0.798935
1,XGBoost,0.263444,0.769678,0.35767,0.839067
2,Gradient Boosting,0.419656,0.809907,0.515262,0.873844
3,MLP,0.369121,0.753243,0.452768,0.840663
4,Linear Regression,0.824853,1.124582,0.938636,1.214111
5,Ridge,0.824863,1.124588,0.938645,1.214118


Node2Vec v1 (128-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.18056,0.637391,0.22062,0.713651
1,XGBoost,0.227172,0.688238,0.313251,0.765946
2,Gradient Boosting,0.283858,0.695074,0.369611,0.775445
3,MLP,0.286074,0.801616,0.379043,0.900057
4,Ridge,0.827239,1.076182,0.9589,1.197344
5,Linear Regression,0.827224,1.076172,0.958884,1.197329


## Hyperparameter Tuning

In [13]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([
        np.full(len(trainer.train_df), -1),
        np.zeros(len(trainer.val_df), dtype=int),
    ])
    search = RandomizedSearchCV(
        base_model, param_distributions,
        n_iter=n_iter, cv=PredefinedSplit(split_idx),
        scoring="neg_root_mean_squared_error",
        random_state=42, n_jobs=-1,
    )
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    return search.best_params_

RF_PARAMS = {
    "model__n_estimators":      [100, 200, 300],
    "model__max_depth":         [None, 5, 10],
    "model__min_samples_leaf":  [1, 2, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10, 15, 20],
    "model__max_features":      ["sqrt", "log2", 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    "model__max_iter":          [100, 200, 300],
    "model__max_depth":         [3, 5, 8, None],
    "model__learning_rate":     [0.005, 0.01, 0.05],
    "model__min_samples_leaf":  [5, 10, 20, 50, 100],
    "model__l2_regularization": [1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__max_leaf_nodes":    [15, 20, 30, 40, 50, 60],
    "model__max_bins":          [64, 128, 255],
}
XGB_PARAMS = {
    "model__n_estimators":     [100, 200, 400],
    "model__max_depth":        [3, 4, 5, 6, 8, 10],
    "model__learning_rate":    [0.005, 0.01, 0.05],
    "model__subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "model__min_child_weight": [1, 2, 5, 10],
    "model__gamma":            [0, 0.1, 0.5, 1.0, 2.0],
    "model__reg_alpha":        [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__reg_lambda":       [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
}
MLP_PARAMS = {
    "model__hidden_layer_sizes": [(64,), (128,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    "model__activation":         ["relu", "tanh"],
    "model__alpha":              [1e-5, 1e-4, 1e-3, 1e-2, 0.1],
    "model__learning_rate_init": [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 0.05],
    "model__learning_rate":      ["constant", "adaptive"],
    "model__batch_size":         [32, 64, 128, "auto"],
}

In [14]:
for label, t in [
    ("GraphSAGE v1 (32-dim)",  trainer_sage_32),
    ("GraphSAGE v1 (64-dim)",  trainer_sage_64),
    ("GraphSAGE v1 (128-dim)", trainer_sage_128),
    ("Node2Vec v1 (32-dim)",   trainer_n2v_32),
    ("Node2Vec v1 (64-dim)",   trainer_n2v_64),
    ("Node2Vec v1 (128-dim)",  trainer_n2v_128),
]:
    print(f"\n===== Tuning {label} =====")
    tune(t, make_pipeline(RandomForestRegressor(random_state=42)),         RF_PARAMS,  "Random Forest (tuned)")
    tune(t, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS,  "Gradient Boosting (tuned)")
    tune(t, make_pipeline(XGBRegressor(random_state=42)),                  XGB_PARAMS, "XGBoost (tuned)")
    tune(t, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=5, random_state=42)), MLP_PARAMS, "MLP (tuned)")
    display(t.leaderboard()[DISPLAY_COLS])
    display(t.leaderboard()[TOP1_COLS])


===== Tuning GraphSAGE v1 (32-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,MLP (tuned),0.020455,0.025791,0.096156,0.121924
1,MLP,0.019889,0.027906,0.08647,0.127898
2,Random Forest,0.005616,0.024808,0.031387,0.138866
3,Random Forest (tuned),0.007663,0.024476,0.043634,0.139102
4,Gradient Boosting (tuned),0.014653,0.024358,0.078915,0.144131
5,XGBoost (tuned),0.014887,0.024892,0.077301,0.145652
6,Gradient Boosting,0.01018,0.025387,0.055033,0.1561
7,XGBoost,0.009549,0.026002,0.050508,0.157843
8,Ridge,0.031269,0.039357,0.111204,0.170686
9,Linear Regression,0.031269,0.039358,0.111204,0.170688


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,MLP (tuned),0.626845,0.676796,0.755482,0.801696
1,MLP,0.484523,0.713926,0.591542,0.832469
2,Random Forest,0.167353,0.836358,0.205482,0.958688
3,Random Forest (tuned),0.242862,0.836913,0.303394,0.961942
4,Gradient Boosting (tuned),0.466108,0.890035,0.561408,1.006387
5,XGBoost (tuned),0.472775,0.908857,0.567391,1.018806
6,Gradient Boosting,0.30016,0.968652,0.376001,1.099647
7,XGBoost,0.26941,0.986194,0.342126,1.113119
8,Ridge,0.626121,0.970383,0.780094,1.109962
9,Linear Regression,0.626107,0.970379,0.78008,1.109958



===== Tuning GraphSAGE v1 (64-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,MLP (tuned),0.027404,0.029428,0.126743,0.1342
1,Random Forest (tuned),0.019021,0.02535,0.108423,0.154471
2,Ridge,0.034163,0.037978,0.133128,0.160703
3,Linear Regression,0.034164,0.037982,0.133128,0.160712
4,Random Forest,0.006826,0.028328,0.038976,0.164166
5,Gradient Boosting (tuned),0.022703,0.030038,0.118978,0.174577
6,XGBoost (tuned),0.019509,0.028167,0.108804,0.17868
7,Gradient Boosting,0.011686,0.029116,0.064773,0.188563
8,XGBoost,0.010257,0.029695,0.055726,0.194917
9,MLP,0.026224,0.03672,0.126762,0.207075


...

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest (tuned),0.008212,0.02771,0.039507,0.126822
1,Random Forest,0.006424,0.027934,0.031188,0.127446
2,MLP (tuned),0.017476,0.032525,0.065525,0.128092
3,XGBoost,0.011768,0.028048,0.053679,0.130683
4,XGBoost (tuned),0.011325,0.028084,0.05044,0.13228
5,Gradient Boosting,0.01562,0.028518,0.071728,0.132775
6,Gradient Boosting (tuned),0.015465,0.028215,0.072954,0.133125
7,MLP,0.025408,0.041764,0.069304,0.135194
8,Linear Regression,0.054328,0.075092,0.12928,0.201878
9,Ridge,0.054326,0.07509,0.12928,0.201878


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest (tuned),0.218729,0.73969,0.27603,0.802923
1,Random Forest,0.17626,0.733163,0.215209,0.798935
2,MLP (tuned),0.346554,0.72797,0.439514,0.805067
3,XGBoost,0.263444,0.769678,0.35767,0.839067
4,XGBoost (tuned),0.248873,0.797993,0.332669,0.861443
5,Gradient Boosting,0.419656,0.809907,0.515262,0.873844
6,Gradient Boosting (tuned),0.420437,0.806636,0.518249,0.870586
7,MLP,0.369121,0.753243,0.452768,0.840663
8,Linear Regression,0.824853,1.124582,0.938636,1.214111
9,Ridge,0.824863,1.124588,0.938645,1.214118



===== Tuning Node2Vec v1 (128-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest (tuned),0.010062,0.027549,0.050526,0.121472
1,Random Forest,0.006331,0.027654,0.031798,0.121654
2,Gradient Boosting (tuned),0.01325,0.029667,0.061037,0.129386
3,XGBoost (tuned),0.011,0.029298,0.049186,0.129817
4,XGBoost,0.010873,0.029189,0.048699,0.130537
5,Gradient Boosting,0.011745,0.029,0.055493,0.130588
6,MLP (tuned),0.021481,0.035582,0.069687,0.131577
7,MLP,0.021324,0.043428,0.058999,0.14532
8,Ridge,0.053409,0.08326,0.128638,0.193767
9,Linear Regression,0.053411,0.083262,0.128638,0.193767


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest (tuned),0.285866,0.646848,0.359363,0.721329
1,Random Forest,0.18056,0.637391,0.22062,0.713651
2,Gradient Boosting (tuned),0.322413,0.696796,0.414957,0.772536
3,XGBoost (tuned),0.24241,0.735417,0.325165,0.804393
4,XGBoost,0.227172,0.688238,0.313251,0.765946
5,Gradient Boosting,0.283858,0.695074,0.369611,0.775445
6,MLP (tuned),0.372104,0.724895,0.472126,0.815185
7,MLP,0.286074,0.801616,0.379043,0.900057
8,Ridge,0.827239,1.076182,0.9589,1.197344
9,Linear Regression,0.827224,1.076172,0.958884,1.197329
